<a href="https://colab.research.google.com/github/Naildelyn/sistemas-opertativos-2026-2/blob/main/sistemas_operativos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import  sys

In [ ]:
def MostrarPCB(Pid, Etiqueta: str):
  #Dos espacio s
  Campos = [
        "Name",
        "State",
        "Pid",
        "PPid",
        "VmSize",
        "VmRSS",
        "voluntary_ctxt_switches",
    ]

  print(f"Mostrando PCB de {Etiqueta}")

  for campo in Campos:
    with open(f"/proc/{Pid}/status") as F:
      for Linea in F:
        if Linea.startswith(campo):
          print(f"{Linea }")


In [2]:
import os

def MostrarPCB(Pid, Etiqueta: str):
  #Dos espacio s
  Campos = [
        "Name",
        "State",
        "Pid",
        "PPid",
        "VmSize",
        "VmRSS",
        "voluntary_ctxt_switches",
    ]

  print(f"Mostrando PCB de {Etiqueta}")

  for campo in Campos:
    with open(f"/proc/{Pid}/status") as F:
      for Linea in F:
        if Linea.startswith(campo):
          print(f"{Linea }")

print ("EJERCICIO 1 - Inspección de la PCB de padre e hijo")

print (f"PADRE PID {os.getpid()}")
MostrarPCB(os.getpid(), "PROCESO PADRE")

EJERCICIO 1 - Inspección de la PCB de padre e hijo
PADRE PID 5485


NameError: name 'MostrarPCB' is not defined

In [ ]:
print(" Creando Proceso Hijo")
Pid = os.fork()
print(f"Estado {Pid}")

if Pid == 0:
  print("Soy el hijo")
  print(f"[HIJO] PID{os.getpid()}")
  print(f"[HIJO] PPID{os.getpid()}")
  print(f" [HIJO] Mostrando mi PCB...")
  MostrarPCB(os.getpid(), "HIJO")

  sys.exit(0) #EL PROCESO SE ACABE O MUERA

else :
  print(f"[PADRE] soy el padre de {Pid}")

os.waitpid(Pid,0) #Cosecha al hijo y libera la PCB


 Creando Proceso Hijo
Estado 0
Soy el hijo
[HIJO] PID5507
[HIJO] PPID5507
 [HIJO] Mostrando mi PCB...


NameError: name 'MostrarPCB' is not defined

 Creando Proceso Hijo
Estado 5507
[PADRE] soy el padre de 5507


/tmp/ipykernel_5485/2981310742.py:2: DeprecationWarning: This process (pid=5485) is multi-threaded, use of fork() may lead to deadlocks in the child.
  Pid = os.fork()


**  Ejercicio 2 - Proceso Zombie**

In [ ]:
import time
import sys
import os

def MostrarPCB(Pid, Etiqueta):
    with open(f"/proc/{Pid}/status") as F:
      for Linea in F:
        if Linea.startswith("State"):
          print(f"[{Etiqueta}] {Linea.strip()}")



Pid = os.fork()

if Pid == 0:
  print(f"[HIJO] PID {os.getpid()}")
  sys.exit(0)

print("[PADRE] Durmiendo por 25 ")
time.sleep(2)
MostrarPCB(Pid, "Zombie Activo")



##Ejercicio 3 - Proceso Huerfano



In [ ]:
Pid = os.fork()

if Pid == 0:

  PpidInicial = os.getppid()
  print(f" [HIJO] PID{os.getpid()} - PPID Inicial: {PpidInicial}")
  print(f" [HIJO] Durmiendo 2s. El padre terminara antes que yo.")
  time.sleep(2)

  PpidFinal = os.getppid()
  print(f"[HIJO] M9i padre ya no existe {PpidInicial}")
  print(f"[HIJO] Me adopto el proceso: {PpidFinal}")
  sys.exit(0)

print(f"[PADRE] PID {os.getpid()} termina antes que el hijo")
print("Sin waitpad el hijo queda huerfano")
print("Se reasigna")


Practica 2

In [ ]:
def RoundRobin(Procesos, q):
    Cola = [{"nombre": P["nombre"],
             "restante": P["burst"]} for P in Procesos]

    tiempo = 0
    espera = {P["nombre"]: for P in Procesos}


    while Cola:
        # Sacamos el primer proceso disponible
        actual = Cola.pop(0)

        # Calculamos cuánto tiempo ejecutará en este turno
        turno = min(q, actual["restante"])
        sobra = actual["restante"] - turno

        print(f"En el tiempo {tiempo}, el proceso {actual['nombre']} "
              f"correrá por {turno} ms, y le faltan {sobra} ms para terminar.")

        actual["restante"] = sobra
        tiempo += turno

        # Si todavía le falta tiempo, vuelve a la cola
        if actual["restante"] > 0:
            Cola.append(actual)
        else:
            print(f"El proceso {actual['nombre']} terminó en el tiempo {tiempo}.\n")


promedio = sum(espera.values()) / len(espera)
print(f"El tiempo promedio de espera de los procesos fue de {promedio} ms")


Procesos = [
    {"nombre": "P1", "burst": 8},
    {"nombre": "P2", "burst": 3},
    {"nombre": "P3", "burst": 2},
]

RoundRobin(Procesos, q=2)

Ejercicio de memoria comportida

In [ ]:
import multiprocessing
import time
import random

def productor(memoria, semaforo):
    for i in range(5):
      valor = random.randint(1,1000)
      semaforo.acquire()
      memoria[i] = valor
      semaforo.release()
      time.sleep(0.15)

def consumidor(memoria, semaforo):
    for i in range(5):
      semaforo.acquire()
      print(f"Consumidor lee:{memoria[i]} en {i}")
      semaforo.release()

memoria = multiprocessing.Array("i",[0]*5)
semaforo = multiprocessing.Semaphore (1)
#0 TODOS pueden acceder
#1 Solo uno puede operar
#2 mutex, coordinados


ProcProductor = multiprocessing.Process(
        target=productor, args=(memoria, semaforo)
    )

ProcConsumidor = multiprocessing.Process(
        target=consumidor, args=(memoria, semaforo)
    )

ProcProductor.start()
ProcConsumidor.start()
ProcProductor.join()
ProcConsumidor.join()
